# Regression Trees: Quality & Commercial Score

Trains a `DecisionTreeRegressor` for each target. No feature scaling needed.
Saves the fitted models and results to `results/`.

## About Decision Trees

A decision tree predicts by asking a series of yes/no questions about a game's features, like a flowchart. "Is the complexity above 3?" If yes, go left; if no, go right. "Does it have worker placement?" Left or right again. After a chain of these questions, you land on a final prediction: the average score of all training games that answered the same way. The tree learns which questions to ask and where to set the thresholds by looking at the training data and finding the splits that group similar scores together. It's easy to understand and interpret, but a single tree can be fragile: small changes in the data can produce a completely different tree.

![Decision Tree](../assets/diagram_decision_tree.png)

## Constants

In [16]:
# ── Constants ──────────────────────────────────────────────────────────────
import os

# How much data goes to training vs. testing.
# Default is 80/20: the model learns from 80% of games (~17,500) and we test
# on the remaining 20% (~4,400). We also run every notebook at 50/50, 70/30,
# and 90/10 to check that results stay consistent (see results/*.md).
# Configurable via environment variable so the run-notebooks script can sweep
# through all splits automatically.
TRAIN_RATIO = float(os.environ.get("LUDOMETRICS_TRAIN_RATIO", "0.80"))
TEST_RATIO = round(1.0 - TRAIN_RATIO, 10)

# Fixed seed so every run produces identical results.
# Without this, the random train/test split would change each time,
# making it impossible to compare models fairly.
RANDOM_STATE = 42

# How many levels of yes/no questions the tree can ask.
# A decision tree works by splitting data: "is playtime > 60 min?" → yes/no,
# then "is complexity > 3?" → yes/no, and so on. Each split adds a level.
# Depth 10 means at most 10 questions in a row before making a prediction.
# Too deep (unlimited) and the tree memorises the training data instead of
# learning general patterns. Too shallow (3–5) and it can't capture enough
# detail. 10 is a common starting point for datasets with hundreds of features.
MAX_DEPTH = 10

SPLIT_LABEL = f"{int(TRAIN_RATIO * 100)}_{int(TEST_RATIO * 100)}"
SPLIT_DISPLAY = f"{int(TRAIN_RATIO * 100)}/{int(TEST_RATIO * 100)}"

## Prepare data

In [17]:
import sys
import time

sys.path.insert(0, "..")

import joblib
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

from utils.train_utils import update_results_table

df = pd.read_csv("../data/games_processed.csv")
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

DROP_COLS = ["BGGId", "quality_score", "commercial_score"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
TARGETS = ["quality_score", "commercial_score"]

X = df[FEATURE_COLS]
print(f"Features: {len(FEATURE_COLS)} columns")

X_train, X_test = train_test_split(X, test_size=TEST_RATIO, random_state=RANDOM_STATE)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Loaded 21,925 rows × 403 columns
Features: 400 columns
Train: 17,540  |  Test: 4,385


## Train one model per target

For each target score, we give the tree all 17,500 training games and let it figure out the best sequence of yes/no questions. At each step, the tree picks the feature and threshold that best separates high-scoring games from low-scoring ones, for example "is complexity above 2.8?" It keeps splitting until it reaches our depth limit of 10 levels. After training, we can see how complex the tree ended up: how many levels deep it goes and how many final prediction groups (leaves) it created.

A tree of depth 10 can have at most 1,024 leaves (2 ^ 10 = 1,024), but ours ends up with fewer because many branches get cut short when splitting further didn't help separate scores. Each game follows just one path through the tree, answering at most 10 yes/no questions to land on one of those groups.

In [18]:
results = {}

for target in TARGETS:
    y_train = df.loc[X_train.index, target]

    model = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
    t0 = time.monotonic()
    model.fit(X_train, y_train)
    elapsed = time.monotonic() - t0

    results[target] = {"model": model, "elapsed": elapsed}
    print(f"{target:20s}  trained in {elapsed:.1f}s")

# Show the tree structure for each target
for target in TARGETS:
    tree = results[target]["model"]
    max_possible = 2 ** tree.get_depth()
    print(f"\n{target} tree structure:")
    print(f"  Depth:  {tree.get_depth()} levels of questions")
    print(f"  Leaves: {tree.get_n_leaves()} prediction groups (out of {max_possible:,} possible at depth {tree.get_depth()})")

quality_score         trained in 0.2s
commercial_score      trained in 0.2s

quality_score tree structure:
  Depth:  10 levels of questions
  Leaves: 395 prediction groups (out of 1,024 possible at depth 10)

commercial_score tree structure:
  Depth:  10 levels of questions
  Leaves: 424 prediction groups (out of 1,024 possible at depth 10)


## Evaluate

To make a prediction, the tree runs a new game through its chain of yes/no questions until it lands on a leaf. The leaf's value (the average score of training games that landed there) becomes the prediction. We test this on a single well-known game first to see it in action, then measure RMSE and R² across all 4,400 test games.

In [19]:
# Load game names for readable output
game_names = pd.read_csv("../dataset/games.csv", usecols=["BGGId", "Name"]).set_index("BGGId")["Name"]

# Pick the highest-rated game in the test set as our example
example_idx = df.loc[X_test.index, "quality_score"].idxmax()
example_row = X_test.loc[[example_idx]]
example_id = df.loc[example_idx, "BGGId"]
example_name = game_names.get(example_id, f"BGGId {example_id}")

print(f"Example prediction ({example_name}):")
for target in TARGETS:
    model = results[target]["model"]
    pred = model.predict(example_row)[0]
    actual = df.loc[example_idx, target]
    print(f"  Predicted {target}: {pred:.1f}")
    print(f"  Actual    {target}: {actual:.1f}")
print()

# Evaluate on all test games
for target in TARGETS:
    # Commonly referred to as y_test — the actual target values for the test set
    y_actual = df.loc[X_test.index, target]
    model = results[target]["model"]
    # Commonly referred to as y_pred — the model's predicted values
    y_predicted = model.predict(X_test)
    rmse = mean_squared_error(y_actual, y_predicted) ** 0.5
    r2 = r2_score(y_actual, y_predicted)
    results[target]["rmse"] = rmse
    results[target]["r2"] = r2
    results[target]["y_predicted"] = y_predicted
    results[target]["y_actual"] = y_actual
    print(f"{target:20s}  RMSE={rmse:.4f}  R²={r2:.4f}")

Example prediction (Gloomhaven):
  Predicted quality_score: 82.5
  Actual    quality_score: 85.1
  Predicted commercial_score: 91.4
  Actual    commercial_score: 94.3

quality_score         RMSE=3.0875  R²=0.4003
commercial_score      RMSE=10.7135  R²=0.4565


## Save results

In [20]:
MODEL_DIR = Path("../results/models/regression_trees")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for target in TARGETS:
    out_path = MODEL_DIR / f"{target}_{SPLIT_LABEL}.pkl"
    joblib.dump(results[target]["model"], out_path)
    print(f"Saved {out_path}")

    predictions = pd.DataFrame({
        "BGGId": df.loc[X_test.index, "BGGId"].values,
        "actual": results[target]["y_actual"].values,
        "predicted": results[target]["y_predicted"],
    })
    pred_path = MODEL_DIR / f"{target}_{SPLIT_LABEL}_predictions.csv"
    predictions.to_csv(pred_path, index=False)
    print(f"Saved {pred_path}")

RESULTS_DIR = Path("../results")

for target in TARGETS:
    elapsed = results[target]["elapsed"]
    minutes, seconds = divmod(elapsed, 60)
    time_label = (
        f"{int(minutes)}m {seconds:.1f}s" if minutes >= 1 else f"{seconds:.1f}s"
    )
    update_results_table(
        RESULTS_DIR / f"{target}.md",
        algorithm="Regression Trees (DecisionTreeRegressor)",
        split=SPLIT_DISPLAY,
        train_size=len(X_train),
        test_size=len(X_test),
        training_time=time_label,
        rmse=results[target]["rmse"],
        r2=results[target]["r2"],
    )
    print(f"Updated results/{target}.md")

Saved ../results/models/regression_trees/quality_score_80_20.pkl
Saved ../results/models/regression_trees/quality_score_80_20_predictions.csv
Saved ../results/models/regression_trees/commercial_score_80_20.pkl
Saved ../results/models/regression_trees/commercial_score_80_20_predictions.csv
Updated results/quality_score.md
Updated results/commercial_score.md
